# 第45章 核密度图（kdeplot）

用核密度估计平滑展示分布，并理解带宽和边界的影响。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。


## 适用场景

样本量较充足，希望比较连续分布的整体形状。

## 数据结构

连续数值样本；分组KDE要求每组有足够且不完全相同的数值。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 bw_adjust 参数从 0.9 改为 0.5 或 1.5，观察带宽对密度曲线平滑度的影响
2. 修改 common_norm=False 为 common_norm=True，对比独立归一化与共同归一化的曲线高度
3. 调整 cut 参数从 0 为 3，说明边界延伸对密度估计范围的影响


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

rng = np.random.default_rng(36)
n = 240
orders = pd.DataFrame({
    "category": rng.choice(["办公", "数码", "家居"], n, p=[0.34, 0.38, 0.28]),
    "channel": rng.choice(["自然流量", "广告", "会员"], n, p=[0.42, 0.36, 0.22]),
    "region": rng.choice(["华东", "华南", "华北"], n),
    "order_value": np.clip(rng.normal(260, 72, n), 45, None),
    "items": rng.integers(1, 7, n),
})
orders.loc[orders["category"] == "数码", "order_value"] *= 1.35
orders["satisfied"] = rng.choice(["满意", "一般"], n, p=[0.78, 0.22])

marketing = pd.DataFrame({
    "channel": rng.choice(["搜索", "社交", "会员"], n),
    "visits": rng.integers(80, 850, n),
    "ad_spend": rng.uniform(2, 38, n),
})
marketing["sales"] = (
    45 + marketing["visits"] * 0.16 + marketing["ad_spend"] * 2.4
    + marketing["channel"].map({"搜索": 18, "社交": 8, "会员": 32})
    + rng.normal(0, 28, n)
).clip(10)
marketing["conversion"] = (marketing["sales"] / marketing["visits"]).clip(0.02, 0.5)

daily = pd.DataFrame({
    "date": np.tile(pd.date_range("2026-01-01", periods=12, freq="D"), 3),
    "region": np.repeat(["华东", "华南", "华北"], 12),
})
daily["sales"] = (
    np.tile(np.linspace(110, 190, 12), 3)
    + np.repeat([28, 8, 18], 12)
    + rng.normal(0, 9, 36)
)

sns.set_theme(style="whitegrid", context="notebook")
print("订单样本:", orders.shape, "营销样本:", marketing.shape)


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import scipy

fig, ax = plt.subplots(figsize=(8, 4.3))
sns.kdeplot(data=orders, x="order_value", fill=True, color="#1a73e8", cut=0, ax=ax)
ax.set(title="订单金额核密度", xlabel="客单价（元）", ylabel="密度")
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.kdeplot(data=orders, x="order_value", hue="category", common_norm=False, bw_adjust=0.9, cut=0, linewidth=2, palette="colorblind", ax=ax)
ax.set(title="品类客单价密度", xlabel="客单价（元）", ylabel="密度")
fig.tight_layout()
plt.show()


## 3. 参数说明

- bw_adjust：带宽
- fill：填充
- common_norm：共同归一化
- cut：边界延伸


## 4. 结果解读

曲线面积表示概率密度，峰高不是样本数；用不同带宽检查峰形稳定性。


## 常见误区

- 小样本使用KDE
- 边界外出现不可能值
- 把平滑峰当作真实分组


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.kdeplot(data=marketing, x="sales", bw_adjust=0.45, color="#188038", ax=axes[0])
axes[0].set(title="较小带宽")
sns.kdeplot(data=marketing, x="sales", bw_adjust=1.6, color="#188038", ax=axes[1])
axes[1].set(title="较大带宽")
fig.tight_layout()
plt.show()


## 本章小结

用核密度估计平滑展示分布，并理解带宽和边界的影响。


### 你已经掌握

- 判断核密度图（kdeplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 样本量较充足，希望比较连续分布的整体形状。 |
| 数据结构 | 连续数值样本；分组KDE要求每组有足够且不完全相同的数值。 |
| 结果解读 | 曲线面积表示概率密度，峰高不是样本数；用不同带宽检查峰形稳定性。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `bw_adjust` | 带宽 |
| `fill` | 填充 |
| `common_norm` | 共同归一化 |
| `cut` | 边界延伸 |


### 需要注意

- 小样本使用KDE
- 边界外出现不可能值
- 把平滑峰当作真实分组


### 完成检查

- [ ] 能判断什么问题适合使用核密度图（kdeplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论
